#Downloading the Dataset Directly by URL

In [9]:
!git clone https://github.com/emanhamed/Houses-dataset.git

Cloning into 'Houses-dataset'...
remote: Enumerating objects: 2166, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 2166 (delta 0), reused 0 (delta 0), pack-reused 2165 (from 1)
Receiving objects: 100% (2166/2166), 176.26 MiB | 36.43 MiB/s, done.
Resolving deltas: 100% (20/20), done.


# loading tabular data

In [10]:
import pandas as pd
import numpy as np

cols = ["bedrooms", "bathrooms", "area", "zipcode", "price"]
df = pd.read_csv(
    "Houses-dataset/Houses Dataset/HousesInfo.txt",
    sep=" ", header=None, names=cols
)
print(df.shape)
df.head()

(535, 5)


,bedrooms,bathrooms,area,zipcode,price
0,4,4.0,4053,85255,869500
1,4,3.0,3343,36372,865200
2,3,4.0,3923,85266,889000
3,5,5.0,4022,85262,910000
4,3,4.0,4116,85266,971226


# Loading and Match Images to Each House

In [11]:
import cv2
import os

IMG_DIR = "Houses-dataset/Houses Dataset"
IMG_SIZE = 64  # each tile size; final montage will be 128x128

def load_house_images(house_id, img_dir=IMG_DIR, size=IMG_SIZE):
    types = ["bathroom", "bedroom", "frontal", "kitchen"]
    images = []
    for t in types:
        path = os.path.join(img_dir, f"{house_id}_{t}.jpg")
        img = cv2.imread(path)
        img = cv2.resize(img, (size, size))
        images.append(img)

    # arrange into a 2x2 grid -> 128x128x3 composite image
    top = np.hstack([images[0], images[1]])
    bottom = np.hstack([images[2], images[3]])
    montage = np.vstack([top, bottom])
    return montage

# Build full image array aligned with df rows
images = []
valid_idx = []

for i in range(1, len(df) + 1):  # house IDs are 1-indexed
    try:
        img = load_house_images(i)
        images.append(img)
        valid_idx.append(i - 1)  # align to df row index
    except Exception:
        continue  # skip houses with missing images

images = np.array(images, dtype="float32") / 255.0
df = df.iloc[valid_idx].reset_index(drop=True)

print(images.shape)  # (num_houses, 128, 128, 3)
print(df.shape)

(535, 128, 128, 3)
(535, 5)


Since each house has 4 images, the standard approach (from the original paper) is to tile the 4 images into one composite image per house this gives the CNN one combined visual input per row of tabular data.

#  Preprocessing Tabular Data

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# One-hot encode zip code, scale numeric features
df["zipcode"] = df["zipcode"].astype(str)
zip_dummies = pd.get_dummies(df["zipcode"], prefix="zip")

numeric_features = df[["bedrooms", "bathrooms", "area"]]
scaler = MinMaxScaler()
numeric_scaled = pd.DataFrame(
    scaler.fit_transform(numeric_features), columns=numeric_features.columns
)

tabular_features = pd.concat([numeric_scaled, zip_dummies], axis=1).values
target = df["price"].values

# Scale target too — helps training stability
price_scaler = MinMaxScaler()
target_scaled = price_scaler.fit_transform(target.reshape(-1, 1)).flatten()

# Train/Test Split

In [13]:
(train_img, test_img,
 train_tab, test_tab,
 train_y, test_y) = train_test_split(
    images, tabular_features, target_scaled,
    test_size=0.2, random_state=42
)

Same Split for Both Modalities

# Building the CNN Branch

In [14]:
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, Flatten, Dense,
    BatchNormalization, Dropout, concatenate
)
from tensorflow.keras.models import Model

def build_cnn(input_shape=(128, 128, 3)):
    inputs = Input(shape=input_shape)
    x = Conv2D(16, (3, 3), padding="same", activation="relu")(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling2D(pool_size=(2, 2))(x)

    x = Conv2D(32, (3, 3), padding="same", activation="relu")(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D(pool_size=(2, 2))(x)

    x = Conv2D(64, (3, 3), padding="same", activation="relu")(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D(pool_size=(2, 2))(x)

    x = Flatten()(x)
    x = Dense(16, activation="relu")(x)
    x = Dropout(0.3)(x)
    return inputs, x

This is a small custom CNN (fine for 535 samples — a pretrained ResNet would overfit badly on this little data unless heavily frozen). It progressively extracts spatial features and reduces them to a 16-dim feature vector per house.

# Building the Tabular (MLP) Branch

In [15]:
def build_mlp(input_dim):
    inputs = Input(shape=(input_dim,))
    x = Dense(64, activation="relu")(inputs)
    x = Dense(32, activation="relu")(x)
    x = Dense(8, activation="relu")(x)
    return inputs, x

# Fuse Both Branches (Feature Fusion)

In [16]:
cnn_input, cnn_output = build_cnn()
mlp_input, mlp_output = build_mlp(tabular_features.shape[1])

combined = concatenate([cnn_output, mlp_output])

x = Dense(16, activation="relu")(combined)
x = Dense(1, activation="linear")(x)  # regression output

model = Model(inputs=[cnn_input, mlp_input], outputs=x)
model.compile(optimizer="adam", loss="mse", metrics=["mae"])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 128, 128,  │        448 │ input_layer[0][0] │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 128, 128,  │         64 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 64, 64,    │      4,640 │ max_pooling2d[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        128 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 32, 32,    │     18,496 │ max_pooling2d_1[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 16, 16,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 52)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 16384)     │          0 │ max_pooling2d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      3,392 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 16)        │    262,160 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32)        │      2,080 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 16)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 8)         │        264 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 24)        │          0 │ dropout[0][0],    │
│ (Concatenate)       │                   │            │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 16)        │        400 │ concatenate[0][0

 Total params: 292,345 (1.12 MB)

 Trainable params: 292,121 (1.11 MB)

 Non-trainable params: 224 (896.00 B)

: This is the core "multimodal" step — the CNN's 16-dim image embedding and the MLP's 8-dim tabular embedding are concatenated (concatenate) into one combined vector, which is then passed through a final dense layer to predict a single price value. This is the standard late fusion architecture.

# Train

In [18]:
train_tab = train_tab.astype(float)
test_tab = test_tab.astype(float)

history = model.fit(
    x=[train_img, train_tab], y=train_y,
    validation_data=([test_img, test_tab], test_y),
    epochs=50,
    batch_size=8
)

Epoch 1/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 15s 198ms/step - loss: 0.0528 - mae: 0.1085 - val_loss: 26.8191 - val_mae: 5.1705
Epoch 2/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 11s 197ms/step - loss: 0.0160 - mae: 0.0593 - val_loss: 73.0161 - val_mae: 8.5332
Epoch 3/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 12s 223ms/step - loss: 0.0056 - mae: 0.0409 - val_loss: 99.2751 - val_mae: 9.9462
Epoch 4/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 18s 171ms/step - loss: 0.0047 - mae: 0.0316 - val_loss: 89.3997 - val_mae: 9.3984
Epoch 5/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 12s 194ms/step - loss: 0.0042 - mae: 0.0300 - val_loss: 50.1425 - val_mae: 6.9377
Epoch 6/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 20s 197ms/step - loss: 0.0040 - mae: 0.0283 - val_loss: 19.6217 - val_mae: 4.1304
Epoch 7/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 11s 197ms/step - loss: 0.0037 - mae: 0.0281 - val_loss: 4.7083 - val_mae: 1.6245
Epoch 8/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 9s 169ms/step - loss: 0.0037 - mae: 0.0260 - val_loss: 0.6811 - val_mae: 0.3208
Epoch 9/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 11s 1

# Evaluate with MAE and RMSE

In [19]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

preds_scaled = model.predict([test_img, test_tab])

# Inverse-transform back to real dollar values
preds = price_scaler.inverse_transform(preds_scaled)
actual = price_scaler.inverse_transform(test_y.reshape(-1, 1))

mae = mean_absolute_error(actual, preds)
rmse = np.sqrt(mean_squared_error(actual, preds))

print(f"MAE: ${mae:,.2f}")
print(f"RMSE: ${rmse:,.2f}")

4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 407ms/step
MAE: $159,337.67
RMSE: $276,806.02


We inverse-transform predictions back to real dollar scale before computing MAE/RMSE — reporting error in scaled [0,1] units wouldn't be interpretable.

# Compare Against Tabular-Only Baseline

In [20]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(train_tab, train_y)
rf_preds_scaled = rf.predict(test_tab).reshape(-1, 1)
rf_preds = price_scaler.inverse_transform(rf_preds_scaled)

rf_mae = mean_absolute_error(actual, rf_preds)
rf_rmse = np.sqrt(mean_squared_error(actual, rf_preds))

print(f"Tabular-only RF — MAE: ${rf_mae:,.2f}, RMSE: ${rf_rmse:,.2f}")
print(f"Multimodal (CNN+MLP) — MAE: ${mae:,.2f}, RMSE: ${rmse:,.2f}")

Tabular-only RF — MAE: $137,725.70, RMSE: $215,042.73
Multimodal (CNN+MLP) — MAE: $159,337.67, RMSE: $276,806.02


The tabular-only Random Forest baseline outperformed the multimodal CNN+MLP model (MAE $137.7k vs $159.3k; RMSE $215.0k vs $276.8k), which, while counter to the intuitive expectation that adding images improves accuracy, is explained by the small dataset size (535 houses). CNNs typically require far more training examples to learn generalizable visual features, whereas Random Forest is well-suited to small, structured tabular data and resists overfitting through ensembling. This suggests that in this dataset, price-relevant information is already well captured by structured features like zip code and square footage, and the limited image data added noise rather than a useful signal — a finding that highlights an important, general lesson in multimodal ML: additional modalities only help when there's sufficient data to learn from them.